# 03 - Modelagem Gold (Star Schema)

## Contexto

Esta é a camada **Gold**: dados modelados e prontos para responder as perguntas de negócio do objetivo do projeto. A partir da tabela única `silver.steam_games.games`, o notebook monta um **Esquema Estrela**:

- `gold.steam_games.fact_game` — tabela fato: as medidas numéricas de cada jogo (preço, desconto, nota da crítica, aceitação, volume de reviews, popularidade estimada).
- `gold.steam_games.dim_game` — dimensão: os atributos descritivos de cada jogo (título, data de lançamento, faixa etária, plataformas suportadas), incluindo colunas derivadas usadas diretamente pelas perguntas de negócio.
- `gold.steam_games.dim_genre` — dimensão: os gêneros distintos presentes no catálogo.
- `gold.steam_games.bridge_game_genre` — tabela ponte: resolve o relacionamento N:N entre jogo e gênero (um jogo pode ter vários gêneros, um gênero pertence a vários jogos). É a única relação do modelo que não cabe numa dimensão simples.

`fact_game` e `dim_game` compartilham a mesma granularidade (uma linha por jogo) e se relacionam 1:1 por `app_id` — foram separadas para isolar claramente medidas de atributos descritivos, facilitando a leitura do modelo e evitando repetir texto longo (título, datas) em toda consulta agregada.

In [0]:
# Carrega a tabela Silver, que é a base de todas as tabelas Gold.
from pyspark.sql import functions as F
from pyspark.sql.window import Window

silver_games = spark.table("silver.steam_games.games")

## `dim_genre` e `bridge_game_genre`

`genres` chega da Silver como `array<string>`. 228 jogos (0,3%) não têm gênero informado — ficam de fora da tabela ponte (não é erro, é ausência real de dado na fonte).

In [0]:
# dim_genre: transforma a lista de gêneros de cada jogo em uma linha por gênero (explode), mantém só os valores
# distintos e atribui um genre_id sequencial em ordem alfabética (row_number). Grava em
# gold.steam_games.dim_genre e exibe o resultado.
dim_genre = (
    silver_games
    .select(F.explode("genres").alias("genre_name"))
    .filter(F.trim(F.col("genre_name")) != "")
    .distinct()
    .withColumn("genre_id", F.row_number().over(Window.orderBy("genre_name")))
    .select("genre_id", "genre_name")
)

(
    dim_genre.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold.steam_games.dim_genre")
)

print("gold.steam_games.dim_genre:", spark.table("gold.steam_games.dim_genre").count(), "linhas")
display(spark.table("gold.steam_games.dim_genre").orderBy("genre_id"))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


gold.steam_games.dim_genre: 33 linhas


genre_id,genre_name
1,360 Video
2,Accounting
3,Action
4,Adventure
5,Animation & Modeling
6,Audio Production
7,Casual
8,Design & Illustration
9,Documentary
10,Early Access


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1163: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
# bridge_game_genre: explode a lista de gêneros para gerar um par (app_id, genre_name) por jogo e gênero, junta
# com dim_genre pelo nome do gênero para trocá-lo pelo genre_id e mantém só (app_id, genre_id). Resolve a
# relação N:N entre jogos e gêneros.
bridge_game_genre = (
    silver_games
    .select("app_id", F.explode("genres").alias("genre_name"))
    .filter(F.trim(F.col("genre_name")) != "")
    .join(spark.table("gold.steam_games.dim_genre"), on="genre_name", how="inner")
    .select("app_id", "genre_id")
)

(
    bridge_game_genre.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold.steam_games.bridge_game_genre")
)

print("gold.steam_games.bridge_game_genre:", spark.table("gold.steam_games.bridge_game_genre").count(), "linhas")

gold.steam_games.bridge_game_genre: 258243 linhas


## `dim_game`

Atributos descritivos, mais colunas derivadas que as perguntas de negócio usam diretamente: `release_year`/`years_since_release` (pergunta 5), `has_age_restriction` (pergunta 7), `platform_count`/`is_multiplatform` (pergunta 4).

`years_since_release` é calculado em relação à **data de coleta do dataset** (a `release_date` mais recente, 2025-03-10), não à data de hoje: as contagens de reviews ficaram congeladas no momento da coleta, então a idade do jogo precisa ser medida até esse mesmo momento. Usar a data atual deslocaria todos os jogos para faixas mais antigas a cada execução e tornaria o resultado não reprodutível.

In [0]:
# dim_game: seleciona os atributos descritivos do jogo e cria colunas derivadas: ano de lançamento, anos desde o
# lançamento até a data de coleta do dataset (dias divididos por 365,25), has_age_restriction (required_age > 0),
# platform_count (soma das plataformas suportadas) e is_multiplatform (2 ou mais plataformas).
# Data de referência = data de coleta do dataset (a release_date mais recente), não a data de hoje.
data_coleta = silver_games.agg(F.max("release_date")).first()[0]

dim_game = silver_games.select(
    "app_id",
    "title",
    "release_date",
    F.year("release_date").alias("release_year"),
    F.floor(F.datediff(F.lit(data_coleta), F.col("release_date")) / 365.25).alias("years_since_release"),
    "required_age",
    (F.col("required_age") > 0).alias("has_age_restriction"),
    "windows",
    "mac",
    "linux",
    (
        F.col("windows").cast("int") + F.col("mac").cast("int") + F.col("linux").cast("int")
    ).alias("platform_count"),
).withColumn(
    "is_multiplatform", F.col("platform_count") >= 2
)

(
    dim_game.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold.steam_games.dim_game")
)

print("gold.steam_games.dim_game:", spark.table("gold.steam_games.dim_game").count(), "linhas")

gold.steam_games.dim_game: 89729 linhas


## `fact_game`

Medidas numéricas — o que se agrega/compara nas perguntas de negócio.

In [0]:
# fact_game: seleciona as medidas numéricas de cada jogo (preço, desconto, DLCs, nota do Metacritic, aceitação,
# volume de reviews e popularidade estimada), com app_id como chave para ligar com dim_game.
fact_game = silver_games.select(
    "app_id",
    "price",
    "discount",
    "dlc_count",
    "metacritic_score",
    "acceptance_ratio",
    "review_count",
    "review_data_source",
    "estimated_owners_min",
    "estimated_owners_max",
    "estimated_owners_avg",
    "peak_ccu",
)

(
    fact_game.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("gold.steam_games.fact_game")
)

print("gold.steam_games.fact_game:", spark.table("gold.steam_games.fact_game").count(), "linhas")

gold.steam_games.fact_game: 89729 linhas


## Catálogo de Dados

Linhagem de todas as tabelas abaixo: `games_march2025_full.csv` (Kaggle, MIT) → `bronze.steam_games.games_full_raw` (dado bruto) → `silver.steam_games.games` (limpo/tipado) → tabelas Gold abaixo.

### `gold.steam_games.fact_game`
Grão: uma linha por jogo (`app_id`).

| Coluna | Tipo | Domínio | Descrição |
|---|---|---|---|
| `app_id` | int | identificador único da Steam | Chave do jogo (PK, FK para `dim_game`) |
| `price` | double | 0 a ~1000 (USD; máximo 999,98) | Preço atual do jogo |
| `discount` | double | 0 a 100 | Percentual de desconto no momento da coleta |
| `dlc_count` | int | ≥ 0 | Quantidade de DLCs do jogo |
| `metacritic_score` | int | 1 a 100, nulo se sem nota | Nota da crítica especializada (Metacritic). Nulo em ~96% dos jogos |
| `acceptance_ratio` | double | 0 a 100, nulo se sem review | % de avaliações positivas, com fallback entre Steam API e SteamSpy (ver `review_data_source`) |
| `review_count` | double | ≥ 0, nulo se sem review | Volume de reviews, mesma lógica de fallback |
| `review_data_source` | string | `steam_api`, `steamspy` ou nulo | De qual API vieram `acceptance_ratio`/`review_count` nesta linha |
| `estimated_owners_min`/`estimated_owners_max` | long | faixas fixas da fonte (ex. 0-20000, 20000-50000, ...) | Limites da faixa de posse estimada (SteamSpy) |
| `estimated_owners_avg` | double | ponto médio da faixa | Estimativa pontual de popularidade, usada nas perguntas 3 e 6 |
| `peak_ccu` | int | ≥ 0 | Pico de jogadores simultâneos observado |

### `gold.steam_games.dim_game`
Grão: uma linha por jogo (`app_id`).

| Coluna | Tipo | Domínio | Descrição |
|---|---|---|---|
| `app_id` | int | identificador único da Steam | Chave do jogo (PK) |
| `title` | string | texto livre | Nome do jogo. 2 jogos sem nome na fonte original recebem "(nome não informado)" |
| `release_date` | date | 1997-06-30 a 2025-03-10 | Data de lançamento na Steam |
| `release_year` | int | 1997 a 2025 | Ano de lançamento, derivado de `release_date` |
| `years_since_release` | bigint | 0 a 27 | Anos completos entre o lançamento e a data de coleta do dataset (2025-03-10, a `release_date` mais recente), usado na pergunta 5 |
| `required_age` | int | 0, 1, 3, 6, 7, 10, 12, 13, 14, 15, 16, 17, 18, 20, 21; nulo se inválido | Faixa etária mínima exigida. 98,9% dos jogos têm valor 0 (sem restrição) |
| `has_age_restriction` | boolean | true/false | `required_age > 0`, usado na pergunta 7 |
| `windows`/`mac`/`linux` | boolean | true/false | Suporte declarado a cada plataforma |
| `platform_count` | int | 0 a 3 | Quantas das 3 plataformas o jogo suporta |
| `is_multiplatform` | boolean | true/false | `platform_count >= 2`, usado na pergunta 4 |

### `gold.steam_games.dim_genre`
Grão: um gênero distinto.

| Coluna | Tipo | Domínio | Descrição |
|---|---|---|---|
| `genre_id` | int | sequencial, 1 a 33 | Chave substituta do gênero (PK) |
| `genre_name` | string | 33 valores distintos (ex.: Action, Indie, RPG, Simulation...) | Nome do gênero, como declarado na fonte |

### `gold.steam_games.bridge_game_genre`
Grão: um par (jogo, gênero). Resolve o N:N entre `dim_game` e `dim_genre`.

| Coluna | Tipo | Domínio | Descrição |
|---|---|---|---|
| `app_id` | int | FK para `dim_game`/`fact_game` | Jogo |
| `genre_id` | int | FK para `dim_genre` | Gênero associado ao jogo |

Um jogo pode ter zero (228 casos, 0,3%), um ou vários gêneros; um gênero está associado a vários jogos.

## Validação

Contagens esperadas (validadas localmente antes da subida):
- `dim_game` e `fact_game`: **89.729** linhas cada (mesmo grão da Silver)
- `dim_genre`: **33** linhas
- `bridge_game_genre`: **258.243** linhas

In [0]:
# Conta as linhas de cada tabela Gold (esperado: dim_game e fact_game 89.729; dim_genre 33; bridge_game_genre
# 258.243).
display(spark.sql("""
    SELECT 'dim_game' AS tabela, COUNT(*) AS linhas FROM gold.steam_games.dim_game
    UNION ALL
    SELECT 'fact_game', COUNT(*) FROM gold.steam_games.fact_game
    UNION ALL
    SELECT 'dim_genre', COUNT(*) FROM gold.steam_games.dim_genre
    UNION ALL
    SELECT 'bridge_game_genre', COUNT(*) FROM gold.steam_games.bridge_game_genre
"""))

tabela,linhas
dim_game,89729
fact_game,89729
dim_genre,33
bridge_game_genre,258243
